In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification,  get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import time

import torch
import gc


import copy 
import sys

from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from main_transformer import run_simulation_transformer
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity

from data_loader import set_seed

original_stdout = sys.stdout
original_stderr = sys.stderr
import sys

class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def isatty(self):
        return self.terminal.isatty()

    def close(self):
        if self.log:
            self.log.close()
def deep_clean():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
# 
NUM_CLIENTS = 20
MALICIOUS_RATIO = 0.3
GLOBAL_ROUNDS = 15
# 1. Data & Topology

MODEL_CHECKPOINT = "distilbert-base-uncased"
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
train_ds, test_ds = get_data(
    dataset_name='pubmed',
    tokenizer=TOKENIZER
)
test_loader = DataLoader(test_ds, batch_size=256)
client_datasets = distribute_data(train_ds, NUM_CLIENTS)
topology_type = 'scale_free'
G = generate_topology(NUM_CLIENTS, topology_type)
neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}
placement_strategy='Topology-Aware'

num_mal = int(NUM_CLIENTS * MALICIOUS_RATIO)


bf = 0.5
intensity = 0.02


sys.stdout = original_stdout
import time


# ==========================================
# 1. Experiment settup
# ==========================================

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
bf = 1.5
intensity = 1
norm_factor = 15
placement_strategy = 'Topology-Aware'

import pytz
from datetime import datetime


tz = pytz.timezone('Asia/Shanghai')

MALICIOUS_RATIOS = [0.25]

TOPOLOGY_TYPES = [ 'random_regular','scale_free']
MALICIOUS_RATIOS = [0.3, 0.1, 0.2]
DEFENSE_RATIOS = [ 0.2]
SEEDS = [1, 2, 3]
MECHANISMS =  ['MAB',  'FLAME','CosL2', 'FedAvg','TrimmedMean', 'Krum']
NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
SAVE_PATH = ''
for topo_type in TOPOLOGY_TYPES:
    for ratio in MALICIOUS_RATIOS:
        for def_ratio in DEFENSE_RATIOS:

            num_mal = int(NUM_CLIENTS * ratio)
            current_defense_budget = int(NUM_CLIENTS * def_ratio)

            for seed_idx, current_seed in enumerate(SEEDS):
                for mech in MECHANISMS:


                    csv_filename = f"final_Transformer_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_seed{current_seed}.csv"
                    log_filename = f"Log_Transformer_{topo_type}_MR{ratio}_DR{def_ratio}_{mech}_seed{current_seed}.txt"

                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_full_path = os.path.join(SAVE_PATH, log_filename)


                    all_results = []


                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger

                    try:
                        print(f"\n{'='*60}")
                        print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
                        print(f"📡 : Topo={topo_type}, Mal={ratio}, Def={def_ratio}, Seed={current_seed}")
                        print(f"{'='*60}")

                        set_seed(current_seed)
                        print(f"▶️: {mech} | Seed: {current_seed} ({seed_idx+1}/{len(SEEDS)})")


                        G = generate_topology(NUM_CLIENTS, topo_type)
                        neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


                        malicious_clients, defense_nodes = allocate_malicious_nodes(
                            G, num_mal, current_defense_budget, topology_type=topo_type, placement= 'Topology-Aware'
                        )

                        theo_intensities = calculate_theoretical_intensity(
                            neighbors, malicious_clients, NUM_CLIENTS, bf, lambda_benign=0.3
                        )

                        start_tick = time.time()
                        start_wall_time = datetime.now(pytz.timezone('Asia/Shanghai')).strftime("%Y-%m-%d %H:%M:%S")

                        ctx = mp.get_context('spawn')


                        with ProcessPoolExecutor(max_workers=1, mp_context=ctx) as executor:

                            future = executor.submit(
                                run_simulation_transformer,
                                current_seed, NUM_CLIENTS, defense_nodes, malicious_clients,
                                G, neighbors, client_datasets, test_ds,
                                #
                                mechanism=mech, bf=bf, intensity=intensity,
                                debug=False, GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                                norm_factor=norm_factor, epochs=1
                            )

                            #
                            _, _, accs, asrs = future.result()
                        print(f"🚀 Running: {mech}")
                        end_tick = time.time()
                        duration_sec = round(end_tick - start_tick, 2)

                        # 
                        result_entry_base = {
                            'seed': current_seed, 'mechanism': mech, 'malicious_ratio': ratio,
                            'defense_ratio': def_ratio, 'topology': topo_type,
                            'duration_sec': duration_sec
                        }

                        for i in range(NUM_CLIENTS):
                            client_row = copy.deepcopy(result_entry_base)
                            client_row.update({
                                'client_id': i, 'final_acc': accs[i], 'final_asr': asrs[i],
                                'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                                'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                            })
                            all_results.append(client_row)

                        #
                        pd.DataFrame(all_results).to_csv(full_save_path, index=False)

                        #
                        sys.stdout = original_stdout


                        sys.stdout = logger

                    except Exception as e:
                        sys.stdout = original_stdout
                    finally:
                        sys.stdout = original_stdout
                        sys.stderr = original_stderr
                        logger.close()

                    deep_clean()


print(f"\n🎉Experiments compeleted: {SAVE_PATH}")

In [ ]:
#Table 2
# ==========================================
#
# ==========================================
SAVE_PATH = ''
search_pattern = os.path.join(SAVE_PATH, "final_Transformer*.csv")
all_files = glob.glob(search_pattern)

list_df = []

for f in all_files:
    temp_df = pd.read_csv(f)

  
    if "placement_random" in os.path.basename(f):
        temp_df['mechanism'] = 'MAB(RANDOM TOPY)'
        list_df.append(temp_df)

   
    else:
        temp_df['mechanism'] = temp_df['mechanism'].replace({'MAB': 'MAB(TOPY AWARE)'})
        list_df.append(temp_df)


df_results = pd.concat(list_df, ignore_index=True)


df_results['Role'] = df_results['node_type'].apply(lambda x: 'Malicious' if x == 'MAL' else 'Benign')
df_benign = df_results[df_results['Role'] == 'Benign']


benign_comparison = df_benign.pivot_table(
    index='mechanism',
    columns='malicious_ratio',
    values=['final_acc', 'final_asr'],
    aggfunc='mean'
)

print(benign_comparison)

#
df_results2 = df_results[df_results['topology'] ==  'random_regular'].copy()#'scale_free' 'random_regular'
df_results2 = df_results2[df_results2['malicious_ratio'] != 0.4].copy()
df_benign = df_results2[df_results2['Role'] == 'Benign'].copy()

# 2. 
trial_means = df_benign.groupby(
    ['mechanism', 'malicious_ratio', 'seed']
)[['final_acc', 'final_asr']].mean().reset_index()

# 3. 
stats = trial_means.groupby(
    ['mechanism', 'malicious_ratio']
)[['final_acc', 'final_asr']].agg(['mean', 'std']).reset_index()

# 4. 
def format_mean_std(row, metric):
    m = row[(metric, 'mean')]
    s = row[(metric, 'std')]
    return f"{m:.2f} ({s:.2f})"

stats['ACC'] = stats.apply(lambda r: format_mean_std(r, 'final_acc'), axis=1)
stats['ASR'] = stats.apply(lambda r: format_mean_std(r, 'final_asr'), axis=1)

# 5.
clean_stats = stats[['mechanism', 'malicious_ratio', 'ACC', 'ASR']].copy()
latex_table = clean_stats.pivot(index='mechanism', columns='malicious_ratio', values=['ACC', 'ASR'])

# 6. 
latex_table = latex_table.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)

def format_mean_std_percent(row, metric):
    m = row[(metric, 'mean')] * 100
    s = row[(metric, 'std')] * 100
    return f"{m:05.2f} ({s:05.2f})"

# 
stats['ACC (%)'] = stats.apply(lambda r: format_mean_std_percent(r, 'final_acc'), axis=1)
stats['ASR (%)'] = stats.apply(lambda r: format_mean_std_percent(r, 'final_asr'), axis=1)

# 
clean_stats = stats[['mechanism', 'malicious_ratio', 'ACC (%)', 'ASR (%)']].copy()
latex_table = clean_stats.pivot(index='mechanism', columns='malicious_ratio', values=['ACC (%)', 'ASR (%)'])

# 
latex_table = latex_table.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)

# 
custom_order = [
    'FedAvg',
    'Krum',
    'TrimmedMean',
    'CosL2',
    'FLAME',
    'MAB(RANDOM TOPY)',
    'MAB(TOPY AWARE)',
]
# 
existing_order = [m for m in custom_order if m in latex_table.index]
other_mechs = [m for m in latex_table.index if m not in custom_order]
final_order = existing_order + other_mechs

latex_table = latex_table.reindex(final_order)

#
try:
    latex_output = latex_table.style.format(escape="latex").to_latex(
        hrules=True,
        multicol_align="c",
        caption="Macro-Averaged Comparative Performance Summary (Random Regular)",
        label="tab:macro_results_random_regular"
    )
except AttributeError:
    latex_output = latex_table.to_latex(
        multicolumn=True,
        multicolumn_format='c',
        booktabs=True,
        escape=False,
        caption="Macro-Averaged Comparative Performance Summary (Random Regular)",
        label="tab:macro_results_random_regular"
    )

print(latex_output)

In [ ]:
#Figure 2a
import matplotlib.pyplot as plt
import seaborn as sns
df_results2 = df_results[df_results['mechanism'] != 'MAB(RANDOM TOPY)'].copy()

if not df_results2.empty and 'duration_sec' in df_results2.columns:
   

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(8, 6))

    all_mechanisms = df_results2['mechanism'].unique().tolist()

    target_first = 'MAB(TOPY AWARE)'

    if target_first in all_mechanisms:
        custom_order = [target_first] + sorted([m for m in all_mechanisms if m != target_first])
    else:
        custom_order = sorted(all_mechanisms)

  
    sns.barplot(
        data=df_results2,
        x='mechanism',
        y='duration_sec',
        order=custom_order, 
        palette='viridis',
        capsize=.1,
        errorbar='sd'
    )

    plt.title('Average Duration by Mechanism (Transformer)', fontsize=14)
    plt.xlabel('Mechanism', fontsize=12)
    plt.ylabel('Duration (seconds)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show() 